## 1. Invictus IR — Enriched Extraction

Extracts all CloudTrail events with enriched fields and saves  (real test set — never used for training).

In [6]:
import json
import pandas as pd
from pathlib import Path

cloudtrail_path = Path("./aws_dataset/CloudTrail")

# ── Attack label map ──────────────────────────
ATTACK_EVENTS = {
    "CreateAccessKey":         "privilege-escalation",
    "CreateLoginProfile":      "privilege-escalation",
    "UpdateLoginProfile":      "privilege-escalation",
    "AttachUserPolicy":        "privilege-escalation",
    "AttachRolePolicy":        "privilege-escalation",
    "AttachGroupPolicy":       "privilege-escalation",
    "PutUserPolicy":           "privilege-escalation",
    "PutRolePolicy":           "privilege-escalation",
    "PutGroupPolicy":          "privilege-escalation",
    "CreatePolicyVersion":     "privilege-escalation",
    "SetDefaultPolicyVersion": "privilege-escalation",
    "AddUserToGroup":          "privilege-escalation",
    "CreateUser":              "persistence",
    "CreateRole":              "persistence",
    "UpdateAssumeRolePolicy":  "persistence",
    "StopLogging":             "defense-evasion",
    "DeleteTrail":             "defense-evasion",
    "UpdateTrail":             "defense-evasion",
    "PutEventSelectors":       "defense-evasion",
    "GetSecretValue":          "credential-access",
    "GetPasswordData":         "credential-access",
    "PutBucketPolicy":         "exfiltration",
    "DeleteBucketPolicy":      "exfiltration",
}

# ── Helpers ────────────────────────────────────────────────────────────────────

def normalise_principal(identity: dict) -> dict:
    id_type = identity.get("type", "unknown")
    if id_type == "IAMUser":
        return {
            "principal_type": "IAMUser",
            "principal_arn":  identity.get("arn"),
            "username":       identity.get("userName"),
        }
    elif id_type == "AssumedRole":
        session = identity.get("sessionContext", {})
        issuer  = session.get("sessionIssuer", {})
        return {
            "principal_type": "AssumedRole",
            "principal_arn":  identity.get("arn"),
            "username":       issuer.get("userName"),
        }
    elif id_type == "AWSService":
        return {
            "principal_type": "AWSService",
            "principal_arn":  None,
            "username":       identity.get("invokedBy"),
        }
    else:
        return {
            "principal_type": id_type or "unknown",
            "principal_arn":  identity.get("arn"),
            "username":       identity.get("userName"),
        }


# Keys to try, in priority order, when extracting the target resource from
# requestParameters.  The first key that exists and has a non-empty value wins.
_TARGET_KEYS = [
    "roleName",        # IAM role operations
    "userName",        # IAM user operations
    "groupName",       # IAM group operations
    "policyArn",       # Attach/detach policy operations
    "bucketName",      # S3 bucket operations  (also appears as "Host" for path-style)
    "secretId",        # Secrets Manager
    "instanceId",      # EC2
    "functionName",    # Lambda
    "trailName",       # CloudTrail
    "keyId",           # KMS
    "dbInstanceIdentifier",  # RDS
]

def extract_target_resource(params) -> str | None:
    """Return the primary resource name/ARN from requestParameters."""
    if not params or not isinstance(params, dict):
        return None
    for key in _TARGET_KEYS:
        val = params.get(key)
        if val:
            return str(val)
    # Fallback: return the first non-empty string value in the dict
    for val in params.values():
        if isinstance(val, str) and val:
            return val
    return None


# ── Main extraction loop ───────────────────────────────────────────────────────

all_events = []

for json_file in sorted(cloudtrail_path.rglob("*.json")):
    with open(json_file) as f:
        data = json.load(f)

    for event in data.get("Records", []):
        identity   = event.get("userIdentity", {})
        principal  = normalise_principal(identity)
        event_name = event.get("eventName", "unknown")
        params     = event.get("requestParameters")
        session    = identity.get("sessionContext", {})
        attrs      = session.get("attributes", {})

        row = {
            "timestamp":          event.get("eventTime"),
            "event_name":         event_name,
            "event_source":       event.get("eventSource"),
            "aws_region":         event.get("awsRegion"),
            "source_ip":          event.get("sourceIPAddress"),
            "error_code":         event.get("errorCode"),
            "label":              1 if event_name in ATTACK_EVENTS else 0,
            "attack_technique":   ATTACK_EVENTS.get(event_name),
            # ── new fields ────────────────────────────────────────────────────
            "read_only":          event.get("readOnly"),
            "user_agent":         event.get("userAgent"),
            "access_key_id":      identity.get("accessKeyId"),
            "mfa_authenticated":  attrs.get("mfaAuthenticated"),
            "target_resource":    extract_target_resource(params),
            "request_params_raw": json.dumps(params) if params else None,
            # ─────────────────────────────────────────────────────────────────
            **principal,
        }
        all_events.append(row)

df_enriched = pd.DataFrame(all_events)
df_enriched["timestamp"] = pd.to_datetime(df_enriched["timestamp"])
df_enriched.sort_values("timestamp", inplace=True)
df_enriched.reset_index(drop=True, inplace=True)

print(f"Shape: {df_enriched.shape}")
print(f"\nNew columns coverage (non-null %):")
new_cols = ["read_only", "user_agent", "access_key_id", "mfa_authenticated", "target_resource"]
for col in new_cols:
    pct = df_enriched[col].notna().mean() * 100
    print(f"  {col:25s}: {pct:.1f}%")

print(f"\nLabel split:  {df_enriched['label'].value_counts().to_dict()}")
print(f"\nread_only distribution:\n{df_enriched['read_only'].value_counts()}")
print(f"\nSample attack rows with target_resource:")
print(df_enriched[df_enriched['label'] == 1][
    ['event_name', 'attack_technique', 'target_resource', 'user_agent']
].drop_duplicates().head(15).to_string())


Shape: (2900, 17)

New columns coverage (non-null %):
  read_only                : 100.0%
  user_agent               : 100.0%
  access_key_id            : 97.1%
  mfa_authenticated        : 23.2%
  target_resource          : 61.2%

Label split:  {0: 2764, 1: 136}

read_only distribution:
read_only
True     2326
False     574
Name: count, dtype: int64

Sample attack rows with target_resource:
          event_name      attack_technique                              target_resource                                                                                                                                                                                                                                                                                   user_agent
87        CreateRole           persistence  stratus-red-team-ec2-get-password-data-role  APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https://www.terraform.io) terraform-provider-aws/3.76.1 (+https://registry.terraform.io/providers/hashi

In [7]:
# ── Session labeling (same ±5 min window as before) ──────────────────────────
WINDOW_MINUTES = 5
df_enriched["session_label"] = df_enriched["label"].copy()

for username, group in df_enriched.groupby("username"):
    attack_times = group[group["label"] == 1]["timestamp"]
    for atk_time in attack_times:
        mask = (
            (df_enriched["username"] == username) &
            (df_enriched["timestamp"] >= atk_time - pd.Timedelta(minutes=WINDOW_MINUTES)) &
            (df_enriched["timestamp"] <= atk_time + pd.Timedelta(minutes=WINDOW_MINUTES))
        )
        df_enriched.loc[mask, "session_label"] = 1

# ── Save ───────────────────────────────────────────────────────────────────────
df_enriched.to_csv("invictus_enriched.csv", index=False)
print("Saved invictus_enriched.csv")
print(f"Shape:   {df_enriched.shape}")
print(f"Columns: {list(df_enriched.columns)}")
print(f"\nEvent-level attacks:   {df_enriched['label'].sum()}")
print(f"Session-level attacks: {df_enriched['session_label'].sum()}")
print(f"\nNull counts:\n{df_enriched.isnull().sum().to_string()}")


Saved invictus_enriched.csv
Shape:   (2900, 18)
Columns: ['timestamp', 'event_name', 'event_source', 'aws_region', 'source_ip', 'error_code', 'label', 'attack_technique', 'read_only', 'user_agent', 'access_key_id', 'mfa_authenticated', 'target_resource', 'request_params_raw', 'principal_type', 'principal_arn', 'username', 'session_label']

Event-level attacks:   136
Session-level attacks: 2597

Null counts:
timestamp                0
event_name               0
event_source             0
aws_region               0
source_ip                0
error_code            2600
label                    0
attack_technique      2764
read_only                0
user_agent               0
access_key_id           84
mfa_authenticated     2226
target_resource       1125
request_params_raw     333
principal_type           0
principal_arn           77
username                42
session_label            0


## 2. Synthetic Data Generation

Generates 600 labeled sessions (200 attack across 10 MITRE ATT&CK chains + 400 benign) and saves  (training set).

In [8]:
import random, string, json, pandas as pd
from datetime import datetime, timezone, timedelta

random.seed(99)

# ── Unchanged attack chain library (same as before) ───────────────────────────
ATTACK_CHAINS = {
    "create_role_attach_managed_policy": [
        {"event_name": "CreateRole",        "event_source": "iam.amazonaws.com", "attack_technique": "persistence",          "read_only": False, "target_key": "role"},
        {"event_name": "AttachRolePolicy",  "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "role"},
    ],
    "create_role_inline_policy": [
        {"event_name": "CreateRole",    "event_source": "iam.amazonaws.com", "attack_technique": "persistence",          "read_only": False, "target_key": "role"},
        {"event_name": "PutRolePolicy", "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "role"},
    ],
    "create_user_accesskey_policy": [
        {"event_name": "CreateUser",       "event_source": "iam.amazonaws.com", "attack_technique": "persistence",          "read_only": False, "target_key": "user"},
        {"event_name": "CreateAccessKey",  "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "user"},
        {"event_name": "AttachUserPolicy", "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "user"},
    ],
    "create_user_console_access": [
        {"event_name": "CreateUser",         "event_source": "iam.amazonaws.com", "attack_technique": "persistence",          "read_only": False, "target_key": "user"},
        {"event_name": "CreateLoginProfile", "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "user"},
        {"event_name": "AttachUserPolicy",   "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "user"},
    ],
    "add_user_to_admin_group": [
        {"event_name": "AddUserToGroup", "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "group"},
    ],
    "update_role_inline_policy": [
        {"event_name": "PutRolePolicy", "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "role"},
    ],
    "create_policy_version": [
        {"event_name": "CreatePolicyVersion",     "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "policy"},
        {"event_name": "SetDefaultPolicyVersion", "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "policy"},
    ],
    "update_assume_role_policy": [
        {"event_name": "UpdateAssumeRolePolicy", "event_source": "iam.amazonaws.com", "attack_technique": "persistence",          "read_only": False, "target_key": "role"},
        {"event_name": "AssumeRole",             "event_source": "sts.amazonaws.com",  "attack_technique": "privilege-escalation", "read_only": False, "target_key": "role"},
    ],
    "full_kill_chain": [
        {"event_name": "CreateRole",       "event_source": "iam.amazonaws.com",            "attack_technique": "persistence",          "read_only": False, "target_key": "role"},
        {"event_name": "AttachRolePolicy", "event_source": "iam.amazonaws.com",            "attack_technique": "privilege-escalation", "read_only": False, "target_key": "role"},
        {"event_name": "GetSecretValue",   "event_source": "secretsmanager.amazonaws.com", "attack_technique": "credential-access",    "read_only": True,  "target_key": "secret"},
        {"event_name": "PutBucketPolicy",  "event_source": "s3.amazonaws.com",             "attack_technique": "exfiltration",         "read_only": False, "target_key": "bucket"},
        {"event_name": "StopLogging",      "event_source": "cloudtrail.amazonaws.com",     "attack_technique": "defense-evasion",      "read_only": False, "target_key": "trail", "error_probability": 0.4},
    ],
    "ec2_password_data": [
        {"event_name": "CreateRole",      "event_source": "iam.amazonaws.com", "attack_technique": "persistence",          "read_only": False, "target_key": "role"},
        {"event_name": "PutRolePolicy",   "event_source": "iam.amazonaws.com", "attack_technique": "privilege-escalation", "read_only": False, "target_key": "role"},
        {"event_name": "GetPasswordData", "event_source": "ec2.amazonaws.com", "attack_technique": "credential-access",    "read_only": True,  "target_key": "instance", "error_probability": 0.9},
    ],
}

RECON_EVENTS = [
    ("GetAccountSummary",             "iam.amazonaws.com",            True),
    ("ListUsers",                     "iam.amazonaws.com",            True),
    ("ListRoles",                     "iam.amazonaws.com",            True),
    ("ListGroups",                    "iam.amazonaws.com",            True),
    ("ListPolicies",                  "iam.amazonaws.com",            True),
    ("GetAccountAuthorizationDetails","iam.amazonaws.com",            True),
    ("ListAttachedUserPolicies",      "iam.amazonaws.com",            True),
    ("ListAttachedRolePolicies",      "iam.amazonaws.com",            True),
    ("ListBuckets",                   "s3.amazonaws.com",             True),
    ("DescribeInstances",             "ec2.amazonaws.com",            True),
    ("ListSecrets",                   "secretsmanager.amazonaws.com", True),
    ("DescribeTrails",                "cloudtrail.amazonaws.com",     True),
    ("GetCallerIdentity",             "sts.amazonaws.com",            True),
    ("ListAccessKeys",                "iam.amazonaws.com",            True),
]

# ── FIX 1 & 2: Expanded benign event pool — includes write ops ────────────────
# Weighted: (event_name, event_source, read_only, weight)
# write-op weight tuned so overall read_only ratio hits ~0.80 matching real data
BENIGN_EVENTS_WEIGHTED = [
    # Read-only (common background noise)
    ("GetBucketLogging",              "s3.amazonaws.com",             True,  4),
    ("GetBucketPolicy",               "s3.amazonaws.com",             True,  4),
    ("GetBucketAcl",                  "s3.amazonaws.com",             True,  3),
    ("DescribeSecurityGroups",        "ec2.amazonaws.com",            True,  5),
    ("DescribeVpcs",                  "ec2.amazonaws.com",            True,  4),
    ("DescribeSubnets",               "ec2.amazonaws.com",            True,  4),
    ("DescribeInstances",             "ec2.amazonaws.com",            True,  5),
    ("GetRegionOptStatus",            "account.amazonaws.com",        True,  2),
    ("DescribeDBInstances",           "rds.amazonaws.com",            True,  3),
    ("ListKeys",                      "kms.amazonaws.com",            True,  4),
    ("DescribeKey",                   "kms.amazonaws.com",            True,  3),
    ("GetParameter",                  "ssm.amazonaws.com",            True,  5),
    ("DescribeInstanceInformation",   "ssm.amazonaws.com",            True,  4),
    ("GetSecretValue",                "secretsmanager.amazonaws.com", True,  4),
    ("ListFunctions",                 "lambda.amazonaws.com",         True,  2),
    ("DescribeLoadBalancers",         "elasticloadbalancing.amazonaws.com", True, 2),
    # IAM read events — benign users routinely inspect roles/users/policies
    # Breaks the pattern where role-xxx/svc-xxx names only appear in attack sessions
    ("GetRole",           "iam.amazonaws.com", True,  3),
    ("GetUser",           "iam.amazonaws.com", True,  3),
    ("ListRolePolicies",  "iam.amazonaws.com", True,  2),
    ("GetRolePolicy",     "iam.amazonaws.com", True,  2),
    ("GetUserPolicy",     "iam.amazonaws.com", True,  1),
    ("ListGroupsForUser", "iam.amazonaws.com", True,  1),
    # FIX 2 — Benign WRITE ops (present in real data, legitimately write=False)
    ("PutParameter",                  "ssm.amazonaws.com",            False, 3),  # SSM config writes
    ("SendCommand",                   "ssm.amazonaws.com",            False, 3),  # SSM run command
    ("StartInstances",                "ec2.amazonaws.com",            False, 2),  # EC2 lifecycle
    ("StopInstances",                 "ec2.amazonaws.com",            False, 2),  # EC2 lifecycle
    ("RebootInstances",               "ec2.amazonaws.com",            False, 1),
    ("ModifyInstanceAttribute",       "ec2.amazonaws.com",            False, 2),
    ("RotateSecret",                  "secretsmanager.amazonaws.com", False, 2),  # routine rotation
    ("PutSecretValue",                "secretsmanager.amazonaws.com", False, 2),  # app secret update
    ("CreateSnapshot",                "ec2.amazonaws.com",            False, 1),
    ("ModifyDBInstance",              "rds.amazonaws.com",            False, 1),
]

# FIX 3 — AssumedRole benign session template (for services/automation)
ASSUMED_ROLE_BENIGN = [
    ("DescribeInstances",   "ec2.amazonaws.com",            True,  5),
    ("GetParameter",        "ssm.amazonaws.com",            True,  5),
    ("GetSecretValue",      "secretsmanager.amazonaws.com", True,  4),
    ("ListKeys",            "kms.amazonaws.com",            True,  3),
    ("SendCommand",         "ssm.amazonaws.com",            False, 3),
    ("PutParameter",        "ssm.amazonaws.com",            False, 2),
    ("DescribeDBInstances", "rds.amazonaws.com",            True,  2),
]

# FIX 1 — Realistic error codes with correct weights
BENIGN_ERROR_CODES = [
    ("ThrottlingException",                  40),  # most common in real data
    ("Client.UnauthorizedOperation",         17),
    ("AccessDenied",                          6),
    ("NoSuchBucketPolicy",                    5),
    ("Client.InvalidRouteTableID.NotFound",   5),
    ("NoSuchPublicAccessBlockConfiguration",  5),
    ("NoSuchWebsiteConfiguration",            4),
    ("NoSuchCORSConfiguration",               4),
    ("NoSuchLifecycleConfiguration",          4),
]
_benign_err_pool  = [code for code, w in BENIGN_ERROR_CODES for _ in range(w)]
ATTACK_ERROR_CODES = ["AccessDenied", "NoSuchEntity", "ThrottlingException", "InvalidParameterValue"]

USER_AGENTS = [
    "aws-cli/2.13.0 Python/3.11.4 Linux/5.15.0 botocore/2.0.0",
    "Boto3/1.28.0 Python/3.10.6 Linux/5.19.0 Botocore/1.31.0",
    "Boto3/1.26.165 Python/3.10.6 Linux/5.19.0-46-generic Botocore/1.29.165",
    "aws-cli/1.29.0 Python/3.9.0 Darwin/22.0.0 botocore/1.31.0",
    "Terraform/1.5.0 aws-sdk-go/1.44.300",
    "console.amazonaws.com",
    "AWS Internal",
]
ATTACKER_UAS = [
    "aws-cli/2.13.0 Python/3.11.4 Linux/5.15.0 botocore/2.0.0",
    "Boto3/1.28.0 Python/3.10.6 Linux/5.19.0 Botocore/1.31.0",
    "python-requests/2.28.0",
]

def rand_str(n=8):   return "".join(random.choices(string.ascii_lowercase, k=n))
def rand_account():  return "".join(random.choices(string.digits, k=12))
def rand_ip():       return f"{random.randint(10,203)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"
def rand_key():      return "AKIA" + "".join(random.choices(string.ascii_uppercase + string.digits, k=16))
def rand_role_key(): return "ASIA" + "".join(random.choices(string.ascii_uppercase + string.digits, k=16))
def rand_resource(kind):
    s = rand_str(6)
    return {"role": f"role-{s}", "user": f"svc-{s}", "group": f"admins-{s}",
            "policy": "arn:aws:iam::aws:policy/AdministratorAccess",
            "secret": f"prod/db/{s}", "bucket": f"data-{s}-bucket",
            "trail": f"mgmt-trail-{s}", "instance": f"i-{rand_str(17)}"}.get(kind, s)
def jitter(lo=2, hi=45): return timedelta(seconds=random.randint(lo, hi))

def _weighted_sample(pool, n):
    """Sample n items from weighted pool [(item, weight), ...]"""
    items  = [x[:-1] if len(x)==4 else x for x in pool]
    weights= [x[-1] for x in pool]
    return random.choices(items, weights=weights, k=n)


# ── Attack session generator (unchanged logic, same as before) ─────────────────
def generate_attack_session(chain_name, recon_events=5, noise_events=3):
    chain      = ATTACK_CHAINS[chain_name]
    account_id = rand_account(); attacker = rand_str(random.randint(4, 12))
    source_ip  = rand_ip(); access_key = rand_key()
    ua         = random.choice(ATTACKER_UAS)
    t          = datetime(2024, random.randint(1,12), random.randint(1,28),
                          random.randint(7,19), 0, 0, tzinfo=timezone.utc)
    resources  = {s["target_key"]: rand_resource(s["target_key"]) for s in chain}
    rows = []

    for name, source, ro in random.sample(RECON_EVENTS, min(recon_events, len(RECON_EVENTS))):
        t += jitter(3, 30)
        rows.append({"timestamp": t.isoformat(), "event_name": name, "event_source": source,
            "aws_region": "us-east-1", "source_ip": source_ip, "error_code": None,
            "label": 0, "attack_technique": None, "read_only": ro, "user_agent": ua,
            "access_key_id": access_key, "mfa_authenticated": random.choice(["true","false"]),
            "target_resource": None, "request_params_raw": None, "principal_type": "IAMUser",
            "principal_arn": f"arn:aws:iam::{account_id}:user/{attacker}",
            "username": attacker, "session_label": 1, "synthetic": True})

    for step in chain:
        t += jitter(5, 60)
        ep = step.get("error_probability", 0)
        ec = random.choice(ATTACK_ERROR_CODES) if random.random() < ep else None
        rows.append({"timestamp": t.isoformat(), "event_name": step["event_name"],
            "event_source": step["event_source"], "aws_region": "us-east-1",
            "source_ip": source_ip, "error_code": ec, "label": 1,
            "attack_technique": step["attack_technique"], "read_only": step["read_only"],
            "user_agent": ua, "access_key_id": access_key, "mfa_authenticated": "false",
            "target_resource": resources[step["target_key"]],
            "request_params_raw": json.dumps({step["target_key"]+"Name": resources[step["target_key"]]}),
            "principal_type": "IAMUser",
            "principal_arn": f"arn:aws:iam::{account_id}:user/{attacker}",
            "username": attacker, "session_label": 1, "synthetic": True})

    for name, source, ro in _weighted_sample(BENIGN_EVENTS_WEIGHTED, noise_events):
        t += jitter(10, 90)
        rows.append({"timestamp": t.isoformat(), "event_name": name, "event_source": source,
            "aws_region": "us-east-1", "source_ip": source_ip, "error_code": None,
            "label": 0, "attack_technique": None, "read_only": ro, "user_agent": ua,
            "access_key_id": access_key, "mfa_authenticated": "false",
            "target_resource": None, "request_params_raw": None, "principal_type": "IAMUser",
            "principal_arn": f"arn:aws:iam::{account_id}:user/{attacker}",
            "username": attacker, "session_label": 1, "synthetic": True})
    return rows


# ── FIX 1+2: Patched IAMUser benign session ────────────────────────────────────
def generate_benign_iamuser(n_events=15):
    account_id = rand_account(); username = rand_str(6)
    source_ip  = rand_ip(); access_key = rand_key()
    ua         = random.choice(USER_AGENTS)
    t          = datetime(2024, random.randint(1,12), random.randint(1,28),
                          random.randint(7,19), 0, 0, tzinfo=timezone.utc)
    rows = []
    for name, source, ro in _weighted_sample(BENIGN_EVENTS_WEIGHTED, n_events):
        t += jitter(5, 120)
        # FIX 1: error rate 12% (up from 5%), weighted toward ThrottlingException
        ec = random.choice(_benign_err_pool) if random.random() < 0.12 else None
        rows.append({"timestamp": t.isoformat(), "event_name": name, "event_source": source,
            "aws_region": "us-east-1", "source_ip": source_ip, "error_code": ec,
            "label": 0, "attack_technique": None, "read_only": ro, "user_agent": ua,
            "access_key_id": access_key, "mfa_authenticated": random.choice(["true","false"]),
            "target_resource": None, "request_params_raw": None, "principal_type": "IAMUser",
            "principal_arn": f"arn:aws:iam::{account_id}:user/{username}",
            "username": username, "session_label": 0, "synthetic": True})
    return rows


# ── FIX 3: AssumedRole benign session (automation/service accounts) ────────────
def generate_benign_assumed_role(n_events=12):
    account_id = rand_account()
    role_name  = random.choice(["AWSServiceRoleForEC2", "LambdaExecutionRole",
                                 "ECSTaskRole", "CodeDeployRole", "AutoScalingRole"])
    session_id = rand_str(16)
    source_ip  = random.choice(["AWS Internal", rand_ip()])
    access_key = rand_role_key()
    ua         = random.choice(["AWS Internal", "aws-sdk-java/1.11.x", "aws-sdk-go/1.44.x"])
    t          = datetime(2024, random.randint(1,12), random.randint(1,28),
                          random.randint(7,19), 0, 0, tzinfo=timezone.utc)
    arn        = f"arn:aws:sts::{account_id}:assumed-role/{role_name}/{session_id}"
    rows = []
    for name, source, ro in _weighted_sample(ASSUMED_ROLE_BENIGN, n_events):
        t += jitter(1, 30)
        ec = random.choice(_benign_err_pool) if random.random() < 0.08 else None
        rows.append({"timestamp": t.isoformat(), "event_name": name, "event_source": source,
            "aws_region": "us-east-1", "source_ip": source_ip, "error_code": ec,
            "label": 0, "attack_technique": None, "read_only": ro, "user_agent": ua,
            "access_key_id": access_key, "mfa_authenticated": "false",
            "target_resource": None, "request_params_raw": None,
            "principal_type": "AssumedRole", "principal_arn": arn,
            "username": role_name, "session_label": 0, "synthetic": True})
    return rows


# ── Generate the fixed dataset ─────────────────────────────────────────────────
N_PER_CHAIN       = 20   # attack sessions per chain type  -> 200 attack sessions
N_BENIGN_IAMUSER  = 340  # ~85% of benign sessions
N_BENIGN_ASSUMED  = 60   # ~15% of benign sessions -> matches real 2.6% AssumedRole

all_rows = []

for chain_name in ATTACK_CHAINS:
    for _ in range(N_PER_CHAIN):
        all_rows.extend(generate_attack_session(
            chain_name,
            recon_events=random.randint(3, 8),
            noise_events=random.randint(2, 5),
        ))

for _ in range(N_BENIGN_IAMUSER):
    all_rows.extend(generate_benign_iamuser(n_events=random.randint(8, 20)))

for _ in range(N_BENIGN_ASSUMED):
    all_rows.extend(generate_benign_assumed_role(n_events=random.randint(6, 15)))

df_fixed = pd.DataFrame(all_rows)
df_fixed["timestamp"] = pd.to_datetime(df_fixed["timestamp"])
df_fixed.sort_values("timestamp", inplace=True)
df_fixed.reset_index(drop=True, inplace=True)

print(f"Shape: {df_fixed.shape}")
print(f"Attack sessions: {N_PER_CHAIN * len(ATTACK_CHAINS)}")
print(f"Benign sessions: {N_BENIGN_IAMUSER + N_BENIGN_ASSUMED}  "
      f"({N_BENIGN_IAMUSER} IAMUser + {N_BENIGN_ASSUMED} AssumedRole)")
print(f"\nLabel split:")
print(f"  Benign  (0): {(df_fixed['label']==0).sum()}")
print(f"  Attack  (1): {(df_fixed['label']==1).sum()}")


Shape: (7861, 19)
Attack sessions: 200
Benign sessions: 400  (340 IAMUser + 60 AssumedRole)

Label split:
  Benign  (0): 7381
  Attack  (1): 480


In [9]:
# ── Validate all 3 fixes against real data ────────────────────────────────────
df_real = pd.read_csv("invictus_enriched.csv")

print("=" * 55)
print("GAP VALIDATION — Fixed vs Real")
print("=" * 55)

checks = [
    ("read_only ratio",
     df_real["read_only"].mean(),
     df_fixed["read_only"].mean(),
     0.80, 0.02),
    ("error_code rate",
     df_real["error_code"].notna().mean(),
     df_fixed["error_code"].notna().mean(),
     0.10, 0.04),
]

for name, real_val, syn_val, target, tolerance in checks:
    gap    = abs(real_val - syn_val)
    status = "PASS" if gap <= tolerance else "NEEDS TUNING"
    print(f"\n{name}")
    print(f"  Real:      {real_val:.3f}")
    print(f"  Synthetic: {syn_val:.3f}  (gap={gap:.3f}, tolerance±{tolerance})  [{status}]")

print(f"\nprincipal_type distribution")
real_pt = df_real["principal_type"].value_counts(normalize=True).round(3)
syn_pt  = df_fixed["principal_type"].value_counts(normalize=True).round(3)
print(f"  Real:      {real_pt.to_dict()}")
print(f"  Synthetic: {syn_pt.to_dict()}")
assumed_pct = syn_pt.get("AssumedRole", 0)
status = "PASS" if assumed_pct > 0.05 else "LOW"
print(f"  AssumedRole coverage: {assumed_pct:.3f}  [{status}]")

print(f"\nerror_code distribution (top 5)")
real_ec = df_real["error_code"].value_counts(normalize=True).head(5).round(3)
syn_ec  = df_fixed["error_code"].value_counts(normalize=True).head(5).round(3)
print(f"  Real:      {real_ec.to_dict()}")
print(f"  Synthetic: {syn_ec.to_dict()}")

print(f"\nwrite op event sources (read_only=False)")
real_ws = df_real[~df_real["read_only"]]["event_source"].value_counts().head(5)
syn_ws  = df_fixed[~df_fixed["read_only"]]["event_source"].value_counts().head(5)
print(f"  Real:      {real_ws.to_dict()}")
print(f"  Synthetic: {syn_ws.to_dict()}")

# ── Save ──────────────────────────────────────────────────────────────────────
df_fixed.to_csv("synthetic_cloudtrail.csv", index=False)
print(f"\nSaved synthetic_cloudtrail.csv  ({df_fixed.shape[0]} rows)")
print("This replaces synthetic_cloudtrail.csv as the training dataset.")


GAP VALIDATION — Fixed vs Real

read_only ratio
  Real:      0.802
  Synthetic: 0.775  (gap=0.027, tolerance±0.02)  [NEEDS TUNING]

error_code rate
  Real:      0.103
  Synthetic: 0.085  (gap=0.018, tolerance±0.04)  [PASS]

principal_type distribution
  Real:      {'IAMUser': 0.948, 'AssumedRole': 0.026, 'unknown': 0.014, 'AWSService': 0.012}
  Synthetic: {'IAMUser': 0.918, 'AssumedRole': 0.082}
  AssumedRole coverage: 0.082  [PASS]

error_code distribution (top 5)
  Real:      {'ThrottlingException': 0.34, 'Client.UnauthorizedOperation': 0.147, 'AccessDenied': 0.053, 'NoSuchBucketPolicy': 0.047, 'Client.InvalidRouteTableID.NotFound': 0.043}
  Synthetic: {'ThrottlingException': 0.433, 'Client.UnauthorizedOperation': 0.201, 'AccessDenied': 0.078, 'NoSuchBucketPolicy': 0.061, 'NoSuchCORSConfiguration': 0.049}

write op event sources (read_only=False)
  Real:      {'ssm.amazonaws.com': 165, 'ec2.amazonaws.com': 155, 'secretsmanager.amazonaws.com': 97, 'iam.amazonaws.com': 88, 's3.amazonaw

## 3. Rule-Based Baseline Evaluation

Evaluates 3 rule sets (Minimal SIEM, GuardDuty-style, Post-incident) on both datasets. GuardDuty-style F1=0.889 is the primary baseline the GNN+Sequence ensemble must surpass.

In [10]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# ── Rule sets ──────────────────────────────────────────────────────────────────
# Each represents a different real-world detection maturity level.
# Source: AWS GuardDuty findings catalogue, Rhino Security Labs IAM list.

RULES = {
    # What a junior analyst puts in a SIEM on day 1 — obvious defense evasion only
    "Minimal SIEM (3 rules)": {
        "StopLogging", "DeleteTrail", "CreateLoginProfile",
    },

    # Approximates GuardDuty IAM findings — covers well-known privilege escalation
    # but NOT AddUserToGroup, UpdateAssumeRolePolicy, or AssumeRole (all legitimate APIs)
    "GuardDuty-style (11 rules)": {
        "CreateLoginProfile", "UpdateLoginProfile",
        "AttachUserPolicy", "AttachRolePolicy", "AttachGroupPolicy",
        "PutUserPolicy", "PutRolePolicy",
        "CreatePolicyVersion", "SetDefaultPolicyVersion",
        "StopLogging", "DeleteTrail",
    },

    # Everything in the known-bad list — what you'd have AFTER reading the incident report
    # (represents post-incident rule tuning, not pre-incident detection)
    "Post-incident rules (all 23)": {
        "CreateAccessKey", "CreateLoginProfile", "UpdateLoginProfile",
        "AttachUserPolicy", "AttachRolePolicy", "AttachGroupPolicy",
        "PutUserPolicy", "PutRolePolicy", "PutGroupPolicy",
        "CreatePolicyVersion", "SetDefaultPolicyVersion", "AddUserToGroup",
        "CreateUser", "CreateRole", "UpdateAssumeRolePolicy",
        "StopLogging", "DeleteTrail", "UpdateTrail", "PutEventSelectors",
        "GetSecretValue", "GetPasswordData", "PutBucketPolicy", "DeleteBucketPolicy",
    },
}

def build_sessions(df, label_col="session_label"):
    """Aggregate events into sessions keyed by username."""
    sessions = {}
    for username, grp in df.groupby("username"):
        sessions[username] = {
            "events":     list(grp["event_name"]),
            "true_label": int(grp[label_col].max()),
        }
    return sessions

def rule_predict(sessions, rule_set):
    return {u: int(any(e in rule_set for e in s["events"])) for u, s in sessions.items()}

def report(sessions, preds, name):
    y_true = [sessions[u]["true_label"] for u in sessions]
    y_pred = [preds[u] for u in sessions]
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)
    tp = sum(1 for u in sessions if preds[u]==1 and sessions[u]["true_label"]==1)
    fp = sum(1 for u in sessions if preds[u]==1 and sessions[u]["true_label"]==0)
    fn = sum(1 for u in sessions if preds[u]==0 and sessions[u]["true_label"]==1)
    missed = [u for u in sessions if preds[u]==0 and sessions[u]["true_label"]==1]
    return {"name": name, "precision": p, "recall": r, "f1": f,
            "tp": tp, "fp": fp, "fn": fn, "missed": missed}

# ── Evaluate on SYNTHETIC (train distribution) ────────────────────────────────
print("=" * 60)
print("EVALUATION ON SYNTHETIC DATA (train distribution)")
print("=" * 60)
df_syn  = pd.read_csv("synthetic_cloudtrail.csv")
sess_syn = build_sessions(df_syn)
print(f"Sessions: {len(sess_syn)}  |  "
      f"Attack: {sum(1 for s in sess_syn.values() if s['true_label']==1)}  |  "
      f"Benign: {sum(1 for s in sess_syn.values() if s['true_label']==0)}\n")

results_syn = []
for rule_name, rule_set in RULES.items():
    preds = rule_predict(sess_syn, rule_set)
    r = report(sess_syn, preds, rule_name)
    results_syn.append(r)
    print(f"{rule_name}")
    print(f"  P={r['precision']:.3f}  R={r['recall']:.3f}  F1={r['f1']:.3f}  "
          f"(TP={r['tp']} FP={r['fp']} FN={r['fn']})")
    if r["missed"]:
        # Show which chain types were missed
        missed_chains = set()
        for u in r["missed"]:
            evts = tuple(e for e in sess_syn[u]["events"]
                         if e not in {"GetAccountSummary","ListUsers","ListRoles",
                                       "ListGroups","ListPolicies","ListBuckets",
                                       "DescribeInstances","ListSecrets","DescribeTrails",
                                       "GetCallerIdentity","ListAccessKeys","GetBucketLogging",
                                       "DescribeSecurityGroups","DescribeVpcs","GetParameter",
                                       "DescribeDBInstances","ListKeys","DescribeKey",
                                       "ListFunctions","GetRegionOptStatus"})
            missed_chains.add(evts)
        print(f"  Missed attack patterns: {[' → '.join(c) for c in missed_chains]}")
    print()

# ── Evaluate on REAL invictus (test set — ground truth) ───────────────────────
print("=" * 60)
print("EVALUATION ON REAL INVICTUS DATA (test set)")
print("=" * 60)
df_real  = pd.read_csv("invictus_enriched.csv")
sess_real = build_sessions(df_real)
print(f"Sessions: {len(sess_real)}  |  "
      f"Attack: {sum(1 for s in sess_real.values() if s['true_label']==1)}  |  "
      f"Benign: {sum(1 for s in sess_real.values() if s['true_label']==0)}\n")

results_real = []
for rule_name, rule_set in RULES.items():
    preds = rule_predict(sess_real, rule_set)
    r = report(sess_real, preds, rule_name)
    results_real.append(r)
    print(f"{rule_name}")
    print(f"  P={r['precision']:.3f}  R={r['recall']:.3f}  F1={r['f1']:.3f}  "
          f"(TP={r['tp']} FP={r['fp']} FN={r['fn']})")
    if r["missed"]:
        print(f"  Missed: {r['missed']}")
    print()

# ── Summary table (for paper) ─────────────────────────────────────────────────
print("=" * 60)
print("PAPER TABLE — Baseline results (synthetic test set)")
print("=" * 60)
print(f"{'Method':<35} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 65)
for r in results_syn:
    print(f"{r['name']:<35} {r['precision']:>10.3f} {r['recall']:>8.3f} {r['f1']:>8.3f}")
print(f"{'GNN + Sequence model (ours)':<35} {'???':>10} {'???':>8} {'???':>8}")
print()
print("NOTE: 'Post-incident rules' represents an unfair upper bound —")
print("these rules were written AFTER reading the incident report.")
print("GuardDuty-style is the fair pre-incident comparison point.")


EVALUATION ON SYNTHETIC DATA (train distribution)
Sessions: 545  |  Attack: 200  |  Benign: 345

Minimal SIEM (3 rules)
  P=1.000  R=0.200  F1=0.333  (TP=40 FP=0 FN=160)
  Missed attack patterns: ['AddUserToGroup → DescribeSubnets → PutSecretValue', 'ListAttachedUserPolicies → AddUserToGroup → StartInstances → ModifyInstanceAttribute → PutParameter → GetUser → StartInstances', 'GetAccountAuthorizationDetails → CreateRole → PutRolePolicy → GetPasswordData → ListRolePolicies', 'ListAttachedRolePolicies → ListAttachedUserPolicies → CreateRole → PutRolePolicy → GetPasswordData → GetSecretValue → GetUser → CreateSnapshot', 'ListAttachedRolePolicies → CreateRole → AttachRolePolicy → DescribeSubnets → StartInstances → GetSecretValue → GetRolePolicy', 'ListAttachedRolePolicies → ListAttachedUserPolicies → CreateRole → AttachRolePolicy → SendCommand → GetRole', 'GetAccountAuthorizationDetails → ListAttachedRolePolicies → CreateRole → PutRolePolicy → GetPasswordData → DescribeSubnets → SendComma